# Model Training — Fussball Vorhersagen
Zwei Modelle: eines mit Aufsteigern, eines ohne.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

print('Libraries geladen!')
print("--")

KeyboardInterrupt: 

## 1. Features laden

In [2]:
df = pd.read_csv('data/features.csv', encoding='utf-8')
print(f'Spiele geladen: {len(df)}')
print(f'Saisons: {df["season"].unique()}')
print(f'Aufsteiger Heimteam: {df["is_promoted_home"].sum()}')
print(f'Aufsteiger Auswärtsteam: {df["is_promoted_away"].sum()}')

Spiele geladen: 610
Saisons: [2024 2025]
Aufsteiger Heimteam: 9
Aufsteiger Auswärtsteam: 10


## 2. Zwei Datensätze erstellen

In [7]:
FEATURES = [
    'home_form', 'away_form', 'form_diff',
    'home_form_home', 'away_form_away',
    'heimquote', 'home_streak', 'away_streak',
    'is_promoted_home', 'is_promoted_away'
]
FEATURES_NO_PROMOTED = [
    'home_form', 'away_form', 'form_diff',
    'home_form_home', 'away_form_away',
    'heimquote', 'home_streak', 'away_streak'
]
TARGET = 'winner'

label_map = {'HOME_TEAM': 1, 'DRAW': 0, 'AWAY_TEAM': -1}
df['target'] = df[TARGET].map(label_map)

# Datensatz A: Alle Spiele
df_all = df.dropna(subset=FEATURES + [TARGET])
print(f'Datensatz A (alle Spiele): {len(df_all)}')

# Datensatz B: Ohne Aufsteiger
df_no_promoted = df[
    (df['is_promoted_home'] == 0) &
    (df['is_promoted_away'] == 0)
].dropna(subset=FEATURES_NO_PROMOTED + [TARGET])
print(f'Datensatz B (ohne Aufsteiger): {len(df_no_promoted)}')
print(f'Rausgelöscht: {len(df_all) - len(df_no_promoted)} Spiele mit Aufsteigern')

KeyError: ['form_diff']

## 3. Train/Test Split für beide Datensätze

In [4]:
# Datensatz A Split
train_a = df_all[(df_all['season'] == 2024) |
                  ((df_all['season'] == 2025) & (df_all['matchday'] <= 17))]
test_a  = df_all[(df_all['season'] == 2025) & (df_all['matchday'] > 17)]

X_train_a = train_a[FEATURES]
y_train_a = train_a['target']
X_test_a  = test_a[FEATURES]
y_test_a  = test_a['target']

print(f'Datensatz A — Training: {len(train_a)} | Test: {len(test_a)}')

# Datensatz B Split
train_b = df_no_promoted[(df_no_promoted['season'] == 2024) |
                          ((df_no_promoted['season'] == 2025) & (df_no_promoted['matchday'] <= 17))]
test_b  = df_no_promoted[(df_no_promoted['season'] == 2025) & (df_no_promoted['matchday'] > 17)]

X_train_b = train_b[FEATURES_NO_PROMOTED]
y_train_b = train_b['target']
X_test_b  = test_b[FEATURES_NO_PROMOTED]
y_test_b  = test_b['target']

print(f'Datensatz B — Training: {len(train_b)} | Test: {len(test_b)}')

NameError: name 'df_all' is not defined

## 4. Baseline

In [ ]:
baseline_a = accuracy_score(y_test_a, [1] * len(y_test_a))
baseline_b = accuracy_score(y_test_b, [1] * len(y_test_b))
print(f'Baseline A (alle Spiele):       {baseline_a*100:.1f}%')
print(f'Baseline B (ohne Aufsteiger):   {baseline_b*100:.1f}%')

## 5. Modell A — Alle Spiele (mit Aufsteigern)

In [ ]:
print('=== Modell A: Alle Spiele ===\n')

lr_a = LogisticRegression(max_iter=1000, random_state=42)
lr_a.fit(X_train_a, y_train_a)
lr_a_acc = accuracy_score(y_test_a, lr_a.predict(X_test_a))
print(f'Logistische Regression: {lr_a_acc*100:.1f}%')

rf_a = RandomForestClassifier(n_estimators=100, random_state=42)
rf_a.fit(X_train_a, y_train_a)
rf_a_pred = rf_a.predict(X_test_a)
rf_a_acc  = accuracy_score(y_test_a, rf_a_pred)
print(f'Random Forest:          {rf_a_acc*100:.1f}%')
print()
print(classification_report(y_test_a, rf_a_pred, target_names=['Auswärtssieg', 'Unentschieden', 'Heimsieg']))

## 6. Modell B — Ohne Aufsteiger

In [ ]:
print('=== Modell B: Ohne Aufsteiger ===\n')

lr_b = LogisticRegression(max_iter=1000, random_state=42)
lr_b.fit(X_train_b, y_train_b)
lr_b_acc = accuracy_score(y_test_b, lr_b.predict(X_test_b))
print(f'Logistische Regression: {lr_b_acc*100:.1f}%')

rf_b = RandomForestClassifier(n_estimators=100, random_state=42)
rf_b.fit(X_train_b, y_train_b)
rf_b_pred = rf_b.predict(X_test_b)
rf_b_acc  = accuracy_score(y_test_b, rf_b_pred)
print(f'Random Forest:          {rf_b_acc*100:.1f}%')
print()
print(classification_report(y_test_b, rf_b_pred, target_names=['Auswärtssieg', 'Unentschieden', 'Heimsieg']))

## 7. Vergleich

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelle = ['Baseline A', 'LR (alle)', 'RF (alle)', 'Baseline B', 'LR (ohne)', 'RF (ohne)']
werte   = [baseline_a, lr_a_acc, rf_a_acc, baseline_b, lr_b_acc, rf_b_acc]
farben  = ['#cccccc', '#2a78d6', '#1baf7a', '#aaaaaa', '#6bb3f5', '#5dd4a8']

axes[0].bar(modelle, [v*100 for v in werte], color=farben)
axes[0].set_title('Genauigkeit der Modelle (%)')
axes[0].set_ylim(0, 100)
axes[0].tick_params(rotation=20)
for i, v in enumerate(werte):
    axes[0].text(i, v*100 + 1, f'{v*100:.1f}%', ha='center', fontsize=8)

# Feature Wichtigkeit bestes RF
best_rf = rf_b if rf_b_acc > rf_a_acc else rf_a
best_features = FEATURES_NO_PROMOTED if rf_b_acc > rf_a_acc else FEATURES
importances = pd.Series(best_rf.feature_importances_, index=best_features).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='#1baf7a')
axes[1].set_title('Feature Wichtigkeit (bester RF)')

plt.tight_layout()
plt.show()

print(f'\nBestes Modell A: RF mit {rf_a_acc*100:.1f}%')
print(f'Bestes Modell B: RF mit {rf_b_acc*100:.1f}%')

## 8. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, y, title in [
    (axes[0], rf_a_pred, y_test_a, 'Modell A (alle Spiele)'),
    (axes[1], rf_b_pred, y_test_b, 'Modell B (ohne Aufsteiger)')
]:
    cm = confusion_matrix(y, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Ausw.', 'Unent.', 'Heim'],
                yticklabels=['Ausw.', 'Unent.', 'Heim'])
    ax.set_title(title)
    ax.set_ylabel('Echtes Ergebnis')
    ax.set_xlabel('Vorhersage')

plt.tight_layout()
plt.show()

## 9. Beide Modelle speichern

In [ ]:
# Modell A speichern (mit Aufsteigern)
with open('data/model_all.pkl', 'wb') as f:
    pickle.dump({'model': rf_a, 'features': FEATURES}, f)
print(f'Modell A gespeichert: data/model_all.pkl ({rf_a_acc*100:.1f}%)')

# Modell B speichern (ohne Aufsteiger)
with open('data/model_no_promoted.pkl', 'wb') as f:
    pickle.dump({'model': rf_b, 'features': FEATURES_NO_PROMOTED}, f)
print(f'Modell B gespeichert: data/model_no_promoted.pkl ({rf_b_acc*100:.1f}%)')

## 10. Manuelle Vorhersage testen

In [ ]:
label_map_reverse = {1: 'Heimsieg', 0: 'Unentschieden', -1: 'Auswärtssieg'}

def vorhersage(home_form, away_form, home_form_home, away_form_away,
               heimquote, home_streak, away_streak,
               is_promoted_home=0, is_promoted_away=0):

    # Modell wählen
    if is_promoted_home == 1 or is_promoted_away == 1:
        model_data = pickle.load(open('data/model_all.pkl', 'rb'))
        modell_name = 'Modell A (mit Aufsteiger)'
    else:
        model_data = pickle.load(open('data/model_no_promoted.pkl', 'rb'))
        modell_name = 'Modell B (ohne Aufsteiger)'

    model    = model_data['model']
    features = model_data['features']

    beispiel = pd.DataFrame([{
        'home_form':        home_form,
        'away_form':        away_form,
        'form_diff':        home_form - away_form,
        'home_form_home':   home_form_home,
        'away_form_away':   away_form_away,
        'heimquote':        heimquote,
        'home_streak':      home_streak,
        'away_streak':      away_streak,
        'is_promoted_home': is_promoted_home,
        'is_promoted_away': is_promoted_away
    }])[features]

    ergebnis = model.predict(beispiel)[0]
    probs    = model.predict_proba(beispiel)[0]

    print(f'Modell: {modell_name}')
    print(f'Vorhersage: {label_map_reverse[ergebnis]}')
    print()
    print('Wahrscheinlichkeiten:')
    for klasse, prob in zip(model.classes_, probs):
        print(f'  {label_map_reverse[klasse]}: {prob*100:.1f}%')

# Beispiel: Bayern vs Dortmund
print('Bayern (Heim) vs Dortmund (Auswärts):')
vorhersage(
    home_form=12, away_form=6,
    home_form_home=10, away_form_away=4,
    heimquote=0.7, home_streak=3, away_streak=0
)

print()
print('Bayern (Heim) vs Hamburger SV (Aufsteiger):')
vorhersage(
    home_form=12, away_form=3,
    home_form_home=10, away_form_away=2,
    heimquote=0.7, home_streak=3, away_streak=0,
    is_promoted_away=1
)